In [1]:
"""
Notebook tái lập kết quả bài toán 1: GNN + Học liên bang cho gợi ý việc làm.

Chạy: jupyter nbconvert --to notebook --execute notebooks/01_medajob_fl.ipynb
hoặc mở bằng Jupyter và chạy lần lượt.

Notebook này KHÔNG huấn luyện lại từ đầu (mất ~80 giây, đã có trong
results/); nó đọc kết quả và tái hiện các bảng/hình chính, kèm phần
chạy lại một thí nghiệm nhỏ để chứng minh pipeline hoạt động.
"""

'\nNotebook tái lập kết quả bài toán 1: GNN + Học liên bang cho gợi ý việc làm.\n\nChạy: jupyter nbconvert --to notebook --execute notebooks/01_medajob_fl.ipynb\nhoặc mở bằng Jupyter và chạy lần lượt.\n\nNotebook này KHÔNG huấn luyện lại từ đầu (mất ~80 giây, đã có trong\nresults/); nó đọc kết quả và tái hiện các bảng/hình chính, kèm phần\nchạy lại một thí nghiệm nhỏ để chứng minh pipeline hoạt động.\n'

# Bài toán 1 — GNN kết hợp Học liên bang cho gợi ý việc làm

Notebook này tái lập các con số trong báo cáo. Cấu trúc:

1. Nạp kết quả đã sinh (`results/reco_main.json`)
2. Bảng so sánh sáu hệ gợi ý
3. Đường cong học liên bang
4. **Chạy lại thật** một thí nghiệm FedAvg nhỏ để chứng minh pipeline sống
5. Kiểm tra tính tất định

In [2]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.common import load_config

cfg = load_config()
results = json.loads((ROOT / "results" / "reco_main.json").read_text(encoding="utf-8"))
print("Đã nạp:", ROOT / "results" / "reco_main.json")
print("Các hệ:", list(results["systems"]))

Đã nạp: E:\2026\DUT\DSS\SauGK\CuoiKi\results\reco_main.json
Các hệ: ['popularity', 'skill_cbf', 'item_knn', 'centralized', 'centralized_sage', 'local_only', 'fedavg', 'fedavg_best', 'fedprox']


## 1. Dữ liệu

In [3]:
d = results["data_summary"]
print(f"Người dùng       : {d['n_users']}")
print(f"Tin tuyển dụng   : {d['n_jobs']:,}")
print(f"Kỹ năng          : {d['n_skills']}")
print(f"Ngành            : {d['n_industries']}")
print(f"Train / Test     : {d['n_train']} / {d['n_test']}")
print(f"Người dùng in test: {d['users_in_test']}")
print(f"Độ thưa          : {d['sparsity_pct']:.2f}%")
print(f"\nChất lượng dữ liệu: {d['quality']['template_jobs']} tin template "
      f"({d['quality']['template_pct']}%)")

Người dùng       : 70
Tin tuyển dụng   : 1,014
Kỹ năng          : 603
Ngành            : 10
Train / Test     : 249 / 58
Người dùng in test: 16
Độ thưa          : 0.43%

Chất lượng dữ liệu: 682 tin template (67.3%)


## 2. Bảng so sánh sáu hệ gợi ý

Con số quan trọng nhất: Recall@20 của mô hình liên bang so với cục bộ
(không chia sẻ gì) và tập trung (mốc trên lý thuyết).

In [4]:
labels = [
    ("popularity", "Phổ biến"),
    ("skill_cbf", "Nội dung (kỹ năng)"),
    ("item_knn", "Item-KNN"),
    ("local_only", "Cục bộ (không chia sẻ)"),
    ("fedavg", "Liên bang (FedAvg)"),
    ("fedprox", "Liên bang (FedProx)"),
    ("centralized", "Tập trung"),
]
rows = []
for key, label in labels:
    s = results["systems"].get(key)
    if not s:
        continue
    ci = s.get("recall@20_ci", {})
    rows.append({
        "Hệ": label,
        "R@10": round(s["recall@10"], 4),
        "R@20": round(s["recall@20"], 4),
        "CI dưới": round(ci.get("lo", float("nan")), 4),
        "CI trên": round(ci.get("hi", float("nan")), 4),
        "NDCG@20": round(s["ndcg@20"], 4),
        "MRR@20": round(s["mrr@20"], 4),
    })
df = pd.DataFrame(rows)
display(df)

,Hệ,R@10,R@20,CI dưới,CI trên,NDCG@20,MRR@20
0,Phổ biến,0.1719,0.2448,0.1146,0.4010,0.1363,0.1372
1,Nội dung (kỹ năng),0.0781,0.1094,0.0000,0.2500,0.0584,0.0773
2,Item-KNN,0.1979,0.3490,0.1771,0.5260,0.1790,0.1385
3,Cục bộ (không chia sẻ),0.3271,0.4021,0.2500,0.5625,0.3103,0.3804
4,Liên bang (FedAvg),0.2917,0.4969,0.3344,0.6552,0.2734,0.2830
5,Liên bang (FedProx),0.3229,0.4479,0.2708,0.6250,0.3118,0.3715
6,Tập trung,0.3281,0.5042,0.3562,0.6605,0.2920,0.2904


In [5]:
loc = results["systems"]["local_only"]["recall@20"]
fed = results["systems"]["fedavg"]["recall@20"]
cent = results["systems"]["centralized"]["recall@20"]
gap = cent - loc
print(f"Cục bộ {loc:.4f} → Liên bang {fed:.4f} → Tập trung {cent:.4f}")
print(f"Học liên bang lấy lại {(fed - loc) / gap:.1%} khoảng cách "
      f"do chia nhỏ dữ liệu gây ra")

Cục bộ 0.4021 → Liên bang 0.4969 → Tập trung 0.5042
Học liên bang lấy lại 92.9% khoảng cách do chia nhỏ dữ liệu gây ra


## 3. Đường cong học liên bang

FedAvg **không hội tụ đơn điệu** — đây là hiện tượng thật, không phải lỗi.

In [6]:
for s in results["client_sweep"]:
    curve = pd.DataFrame(s["curve"])
    print(f"\n=== {s['n_clients']} khách hàng ===")
    print(f"  Cục bộ: {s['local_only']['recall@20']:.4f} | "
          f"FedAvg cuối: {s['fedavg_final']['recall@20']:.4f} | "
          f"đỉnh: {s['fedavg_best']['recall@20']:.4f}")
    display(curve[["round", "recall@20", "ndcg@20", "train_loss"]].head(10))


=== 5 khách hàng ===
  Cục bộ: 0.4021 | FedAvg cuối: 0.4969 | đỉnh: 0.4990


,round,recall@20,ndcg@20,train_loss
0,1,0.357292,0.177609,0.592775
1,2,0.346875,0.252775,0.459314
2,3,0.386458,0.268754,0.292870
3,4,0.415625,0.256351,0.176743
4,5,0.436458,0.272052,0.120621
5,6,0.476042,0.289842,0.089490
6,7,0.488542,0.300584,0.072443
7,8,0.486458,0.306769,0.059441
8,9,0.491667,0.314961,0.054990
9,10,0.498958,0.305155,0.046207



=== 8 khách hàng ===
  Cục bộ: 0.4198 | FedAvg cuối: 0.5031 | đỉnh: 0.5448


,round,recall@20,ndcg@20,train_loss
0,1,0.329167,0.131708,0.590624
1,2,0.379167,0.175438,0.475449
2,3,0.416667,0.192899,0.332440
3,4,0.411458,0.204549,0.219144
4,5,0.432292,0.217501,0.158427
5,6,0.501042,0.250074,0.114972
6,7,0.485417,0.246682,0.098184
7,8,0.485417,0.270978,0.088365
8,9,0.477083,0.289791,0.066153
9,10,0.513542,0.303216,0.066374



=== 12 khách hàng ===
  Cục bộ: 0.2115 | FedAvg cuối: 0.4333 | đỉnh: 0.4333


,round,recall@20,ndcg@20,train_loss
0,1,0.186458,0.108947,0.578400
1,2,0.213542,0.133415,0.470018
2,3,0.250000,0.189146,0.356511
3,4,0.296875,0.163856,0.254957
4,5,0.343750,0.173650,0.188655
5,6,0.356250,0.173798,0.148551
6,7,0.371875,0.179098,0.121427
7,8,0.371875,0.198591,0.107027
8,9,0.384375,0.205993,0.086848
9,10,0.431250,0.226528,0.079305


## 4. Chi phí truyền thông

Đây là kết quả **không thuận lợi**: ở quy mô dữ liệu này, gửi tham số
tốn nhiều byte hơn gửi thẳng dữ liệu thô. Lợi ích của FL là bảo mật.

In [7]:
c = results["communication"]
print(f"Tổng byte gửi lên     : {c['up_bytes_total'] / 1024:,.0f} KB")
print(f"Dữ liệu thô nếu gửi   : {c['raw_data_bytes_if_centralized'] / 1024:.0f} KB")
print(f"Tỉ lệ                 : gấp {c['ratio_up_vs_raw']:,.0f} lần")
print(f"Hoà vốn ở             : {c['crossover_interactions']:,} tương tác")
print(f"\n{c['interpretation']}")

Tổng byte gửi lên     : 19,158 KB
Dữ liệu thô nếu gửi   : 3 KB
Tỉ lệ                 : gấp 6,565 lần
Hoà vốn ở             : 1,634,792 tương tác

FL tốn NHIỀU byte hơn gửi dữ liệu thô ở quy mô này (19158 KB tham số so với 2.9 KB dữ liệu thô) vì payload là O(số nút × số chiều) và không phụ thuộc số tương tác. Lợi ích của FL ở đây KHÔNG phải băng thông mà là dữ liệu gốc không rời khỏi client. FL chỉ rẻ hơn về byte khi tập dữ liệu vượt khoảng 1,634,792 tương tác.


## 5. Chạy lại THẬT một thí nghiệm nhỏ

Phần trên chỉ đọc kết quả. Để chứng minh pipeline hoạt động, ta chạy lại
một cấu hình FedAvg rút gọn (2 vòng, 2 epoch cục bộ) và kiểm tra rằng
mô hình sinh ra đúng loại kết quả.

In [8]:
import numpy as np

from src.medajob.evaluate import evaluate_ranking
from src.medajob.federated import run_federated
from src.medajob.run_experiments import (build_parts, load_artifacts,
                                         scores_from_state)

data = load_artifacts(cfg)
parts, usr_client = build_parts(data, 5)
nU, nJ = data["n_users"], data["n_jobs"]
g = cfg["medajob"]["graph"]


def eval_fn(state):
    s = scores_from_state(state, data, cfg, nU, nJ)
    return evaluate_ranking(s, data["test"], data["Mtr"], k_list=(20,),
                            relevant_threshold=4, n_boot=50, seed=42)


state, logs, info = run_federated(
    parts, data["train"], usr_client, g,
    rounds=2, local_epochs=2, lr=0.02,
    dim=int(g["dim"]), layers=int(g["layers"]),
    reg=1e-5, mu=0.0, sample_rate=1.0, eval_fn=eval_fn, seed=42)

print(f"Sau {info['rounds_run']} vòng rút gọn:")
for lg in logs:
    print(f"  vòng {lg.round}: R@20 = {lg.metrics.get('recall@20', float('nan')):.4f}, "
          f"loss = {lg.mean_train_loss:.4f}, "
          f"gửi lên = {lg.up_bytes:,} byte")
print(f"\nSố tham số           : {info['param_bytes_per_transfer'] // 4:,}")
print(f"Byte mỗi lần chuyển  : {info['param_bytes_per_transfer']:,}")

Sau 2 vòng rút gọn:
  vòng 1: R@20 = 0.0792, loss = 0.6867, gửi lên = 889,700 byte
  vòng 2: R@20 = 0.1052, loss = 0.6782, gửi lên = 884,240 byte

Số tham số           : 54,304
Byte mỗi lần chuyển  : 217,216


## 6. Kiểm tra tính tất định

Hai lần chạy cùng seed phải cho cùng kết quả. Nếu không, mọi so sánh
trong báo cáo đều không đáng tin.

In [9]:
def run_once(seed: int):
    st, lg, _ = run_federated(
        parts, data["train"], usr_client, g,
        rounds=2, local_epochs=2, lr=0.02,
        dim=int(g["dim"]), layers=int(g["layers"]),
        reg=1e-5, mu=0.0, sample_rate=1.0, eval_fn=eval_fn, seed=seed)
    return st["emb.weight"].clone()


a = run_once(42)
b = run_once(42)
c_ = run_once(43)
print(f"|w(seed=42) lần 1 − lần 2| max = {(a - b).abs().max().item():.2e}  "
      f"(phải ~ 0)")
print(f"|w(seed=42) − w(seed=43)| max  = {(a - c_).abs().max().item():.2e}  "
      f"(phải > 0, chứng tỏ seed có tác dụng)")

|w(seed=42) lần 1 − lần 2| max = 4.99e-07  (phải ~ 0)
|w(seed=42) − w(seed=43)| max  = 6.51e-01  (phải > 0, chứng tỏ seed có tác dụng)


## Kết luận

- Học liên bang thu hẹp khoảng cách với mô hình tập trung mà không gửi dữ liệu gốc.
- Mô hình đồ thị vượt mọi baseline: dựng đồ thị có đóng góp thật.
- Ở quy mô này FL tốn nhiều băng thông hơn, lợi ích nằm ở tuân thủ pháp lý.
- Tập kiểm thử chỉ 16 người dùng → khoảng tin cậy rộng, cần thêm dữ liệu.